# Stage 9LF — Late Fusion Attention

All four levels independently trained and frozen.  
Only the **attention MLP** is trained — on a fair playing field.

| Level | Dice | Purity | Faithfulness |
|-------|------|--------|-------------|
| L1 | 0.146 | 0.159 | 0.160 ✅ |
| L2 | 0.336 | 0.569 | 0.218 ✅ |
| L3 | 0.554 | **0.844** | 0.060 ❌ |
| L4 | **0.606** | 0.689 | 0.012 ❌ |

In [ ]:
import sys, csv, time
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# Resolve project root (works whether cwd is project root or notebooks/)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / 'src').exists() else _cwd.parent
assert (PROJECT_ROOT / 'src').exists(), f"Cannot find src/ from {_cwd}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.mmwhs_dataset import MMWHSSliceDataset, make_dataloaders
from src.losses.segmentation import SegmentationLoss, compute_class_weights
from src.metrics.dice import dice_per_class, mean_foreground_dice
from src.models.late_fusion_net import LateFusionProtoNet
from src.models.proto_seg_net_v2 import ProtoSegNetV2

MODALITY    = 'ct'
NUM_CLASSES = 8
LABEL_NAMES = {1:'LV',2:'RV',3:'LA',4:'RA',5:'Myocardium',6:'Aorta',7:'PA'}
LF_LEVELS   = [1, 2, 3, 4]

DATA_DIR   = PROJECT_ROOT / 'data/pack/processed_data'
CKPT_DIR   = PROJECT_ROOT / 'checkpoints'
RESULT_DIR = PROJECT_ROOT / 'results/v9'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LF_CKPT_PATHS = {
    1: CKPT_DIR / 'proto_seg_ct_v2_l1.pth',
    2: CKPT_DIR / 'proto_seg_ct_v2_l2.pth',
    3: CKPT_DIR / 'proto_seg_ct_v2_l3.pth',
    4: CKPT_DIR / 'proto_seg_ct_v2_l4.pth',
}
LF_RESULT_PATH = RESULT_DIR / 'train_curve_proto_ct_v2_lf.csv'
LF_CKPT_SAVE   = CKPT_DIR   / 'proto_seg_ct_v2_lf.pth'

BATCH_SIZE = 16
LR         = 1e-3
MAX_EPOCHS = 80
PATIENCE   = 15
VAL_EVERY  = 5

DEVICE = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')


In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
print('Loading data…')
loaders = make_dataloaders(DATA_DIR, MODALITY, batch_size=BATCH_SIZE)
train_loader = loaders['train']
val_loader   = loaders['val']
print(f'  Train: {len(train_loader.dataset)}  Val: {len(val_loader.dataset)}')

weights_path = PROJECT_ROOT / f'data/class_weights_{MODALITY}.pt'
if weights_path.exists():
    class_weights = torch.load(weights_path, weights_only=True)
else:
    print('  Computing class weights…')
    class_weights = compute_class_weights(DATA_DIR, MODALITY)
    torch.save(class_weights, weights_path)
print('  Class weights loaded')


In [ ]:
# ── Load per-level checkpoints ────────────────────────────────────────────────
print('Loading per-level checkpoints…')
level_models = {}
for l, ckpt_path in LF_CKPT_PATHS.items():
    if l not in LF_LEVELS:
        continue
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    m = ProtoSegNetV2(n_classes=NUM_CLASSES, proto_levels=(l,), use_attention=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    level_models[l] = m
    print(f'  L{l}: best_val_dice={ckpt["best_val_dice"]:.4f}')

model = LateFusionProtoNet(level_models).to(DEVICE)
counts = model.count_parameters()
print(f'\nParams: frozen={counts["frozen"]:,}  trainable_attn={counts["trainable_attn"]:,}')

seg_loss  = SegmentationLoss(class_weights=class_weights.to(DEVICE), n_classes=NUM_CLASSES)
optimizer = torch.optim.AdamW(model.attn_mlp.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def validate():
    model.eval()
    all_logits, all_labels = [], []
    for batch in val_loader:
        logits, _, _ = model(batch['image'].to(DEVICE))
        all_logits.append(logits.cpu())
        all_labels.append(batch['label'])
    model.train()
    return dice_per_class(torch.cat(all_logits), torch.cat(all_labels))

@torch.no_grad()
def mean_attn_weights():
    model.eval()
    w_sum, n = None, 0
    for batch in val_loader:
        _, _, w = model(batch['image'].to(DEVICE))
        wc = w.mean(dim=0).cpu()
        w_sum = wc if w_sum is None else w_sum + wc
        n += 1
    model.train()
    return (w_sum / n).tolist()

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
attn_cols  = [f'attn_w_L{l}' for l in LF_LEVELS]
class_cols = [f'val_dice_{LABEL_NAMES[c]}' for c in range(1, NUM_CLASSES)]
fieldnames = ['epoch','train_loss','val_mean_fg_dice','lr','epoch_time_s'] + attn_cols + class_cols

csv_file = open(LF_RESULT_PATH, 'w', newline='')
writer   = csv.DictWriter(csv_file, fieldnames=fieldnames)
writer.writeheader()

best_val, best_ep, no_improve = 0.0, 0, 0
print(f'Training attention MLP — up to {MAX_EPOCHS} epochs\n')

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t0 = time.time()
    total_loss, n_batches = 0.0, 0

    for batch in train_loader:
        imgs = batch['image'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits, _, _ = model(imgs)
        out = seg_loss(logits, lbls)
        out['loss'].backward()
        nn.utils.clip_grad_norm_(model.attn_mlp.parameters(), 1.0)
        optimizer.step()
        total_loss += out['loss'].item()
        n_batches  += 1

    scheduler.step()
    epoch_time = time.time() - t0
    avg_loss   = total_loss / n_batches
    current_lr = scheduler.get_last_lr()[0]

    row = {'epoch': epoch, 'train_loss': round(avg_loss, 5),
           'lr': round(current_lr, 7), 'epoch_time_s': round(epoch_time, 1)}

    if epoch % VAL_EVERY == 0 or epoch == 1:
        val_dice = validate()
        val_mean = mean_foreground_dice(val_dice)
        row['val_mean_fg_dice'] = round(val_mean, 5)
        for c in range(1, NUM_CLASSES):
            v = val_dice.get(LABEL_NAMES[c], float('nan'))
            row[f'val_dice_{LABEL_NAMES[c]}'] = round(v, 4) if v == v else 'nan'

        w_mean = mean_attn_weights()
        for j, l in enumerate(LF_LEVELS):
            row[f'attn_w_L{l}'] = round(w_mean[j], 5)

        if val_mean > best_val:
            best_val, best_ep, no_improve = val_mean, epoch, 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'best_val_dice': best_val, 'lf_levels': LF_LEVELS}, LF_CKPT_SAVE)
            flag = ' ← best'
        else:
            no_improve += 1; flag = ''

        w_str = '  '.join(f'w_L{l}={row[f"attn_w_L{l}"]:.3f}' for l in LF_LEVELS)
        print(f'  Ep {epoch:3d}/{MAX_EPOCHS} | loss={avg_loss:.4f} | '
              f'val={val_mean:.4f}{flag} | [{w_str}] | {epoch_time:.1f}s', flush=True)
    else:
        row['val_mean_fg_dice'] = ''
        for c in range(1, NUM_CLASSES): row[f'val_dice_{LABEL_NAMES[c]}'] = ''
        for l in LF_LEVELS: row[f'attn_w_L{l}'] = ''

    writer.writerow(row)
    csv_file.flush()

    if no_improve >= PATIENCE:
        print(f'\n  Early stopping at epoch {epoch}')
        break

csv_file.close()
print(f'\nBest val Dice: {best_val:.4f} at epoch {best_ep}')
print(f'Checkpoint:    {LF_CKPT_SAVE}')

In [ ]:
# ── Attention weight evolution + Val Dice ────────────────────────────────────
lf_log  = pd.read_csv(LF_RESULT_PATH)
val_log = lf_log.dropna(subset=['val_mean_fg_dice'])

level_colors = {1: '#BCB22C', 2: '#64B5CD', 3: '#8172B2', 4: '#4C72B0'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for l in LF_LEVELS:
    col = f'attn_w_L{l}'
    if col in val_log.columns:
        ax.plot(val_log['epoch'], val_log[col].astype(float), lw=2,
                label=f'L{l}', color=level_colors[l])
ax.axhline(1/len(LF_LEVELS), color='gray', ls='--', lw=0.8, label='Uniform (0.25)')
ax.set(title='Late Fusion — Attention Weights', xlabel='Epoch',
       ylabel='w_L', ylim=(-0.02, 1.02))
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(val_log['epoch'], val_log['val_mean_fg_dice'].astype(float),
        'o-', ms=4, lw=1.5, color='#DD8452', label='9LF')
for label, val, color in [
    ('9a L4 (0.606)', 0.606, '#4C72B0'),
    ('9L3 L3 (0.554)', 0.554, '#8172B2'),
    ('9L2 L2 (0.336)', 0.336, '#64B5CD'),
]:
    ax.axhline(val, color=color, ls='--', lw=1.2, label=label)
ax.set(title='Late Fusion — Val Dice', xlabel='Epoch', ylabel='Dice')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / 'lf_attn_weights_ct.png', dpi=130)
plt.show()

print('\nFinal attention weights:')
last = val_log.iloc[-1]
for l in LF_LEVELS:
    print(f'  w_L{l} = {float(last[f"attn_w_L{l}"]):.4f}')
print(f'\nBest val Dice: {val_log["val_mean_fg_dice"].astype(float).max():.4f}')